# 08 - Evaluate models without Skewness-Align inputs

Test whether a learned error predictor remains useful without the original method's two inputs.
This notebook retrains the existing LightGBM regressor; its inputs never contain `skew_pred` or
`log_alpha`. The original Skewness-Align score is evaluated separately as an offline reference.

| Default variant | Features | Description |
| --- | ---: | --- |
| `v0_no_skew` | 17 | v0 with `skew_pred` and `log_alpha` removed |
| `v0_no_skew_src_contrast` | 18 | The same inputs plus `src_edge_contrast` |

Default seeds **0, 1, 2** and one tree budget produce **6 fresh model fits**. Each fit is evaluated
under the existing empirical cutoff, a training-fixed-grid empirical control, and a conservative grid policy.
All three policies reuse the same fitted model and do not require additional fits.

This is an **offline answer-quality experiment**. It still reads the logger's precomputed SA scores
for the reference and builds the existing full feature table. Removing model inputs here does not
remove SA computation or source reads from a deployed query path. Source contrast still requires
source histograms. End-to-end latency, actual SA bypass, and raw-data fallback belong to notebook 09.

In [ ]:
%pip install lightgbm==4.6.0

## Settings and reproducible inputs

Use the same query dataset, logger exports, package environment, and query splits as notebooks 06/07.
Required files are `data_root/<dataset>/queries-Ans.txt`, `queries-meta.csv`, and
`sample_root/<dataset>/rows.csv`, `src_*.txt`, `arm_<resolution>.txt`.
Inputs are read-only and copied to a unique personal output directory by default.

- `tree_budgets=4000` preserves the earlier model budget. To compare smaller models explicitly, use
  `128,512,4000`; that increases the default experiment from 6 to 18 fits. Models are all reported;
  no setting is chosen automatically from test outcomes.
- `epsilons=0.25`, `delta=0.05`: an accepted answer violates the target when relative error **>25%**.
- `grid_size=32`, `confidence=0.95`: predeclare the conservative calibration family described below.
- `mlflow_enabled=false` saves files without MLflow; true logs to a personal experiment.
- `copy_inputs=true` saves exact input snapshots. Keep the same inputs across experiments and compare
  `input_manifest.json` fingerprints before interpreting cross-run differences.

Exact answers below 10 are excluded by the existing pipeline. This experiment therefore does not
assess zero-count queries or the population of all range queries.
After changing `repo_root`, rerun the Settings/import cell. Databricks retains old widget values.

In [ ]:
STUDY = 'no-skew'
NOTEBOOK_FILENAME = '08_no_skew_model_eval.ipynb'
DEFAULT_VARIANTS = 'v0_no_skew,v0_no_skew_src_contrast'
BASELINE_VARIANT = 'v0_no_skew'
SCORERS = ('learned', 'skew_align')
POLICIES = ('empirical', 'grid_empirical', 'ucb_grid')

import hashlib
import importlib
import importlib.metadata
import inspect
import json
import math
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import time
from uuid import uuid4

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def _repo_root():
    try:
        notebook = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        root = notebook.split('/ml/notebooks/')[0]
        return root if root.startswith('/Workspace/') else '/Workspace' + root
    except Exception:
        return os.environ.get('REPO_ROOT', os.getcwd())


def _defaults():
    repo = _repo_root()
    output, experiment = '', ''
    parts = Path(repo).parts
    if 'Users' in parts and len(parts) > parts.index('Users') + 1:
        user = parts.index('Users') + 1
        output = str(Path(*parts[:user + 1]) / 'synopsislake-results')
        experiment = f'/Users/{parts[user]}/synopsislake-{STUDY}'
    volume = os.environ.get('VOLUME', '/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake')
    return dict(repo_root=repo, data_root=volume + '/data/ml', sample_root=volume + '/output/ml',
                output_root=output, experiment=experiment, dataset='authors_range', seeds='0,1,2',
                variants=DEFAULT_VARIANTS, epsilons='0.25', delta='0.05', tree_budgets='4000', grid_size='32', confidence='0.95',
                num_threads='2', copy_inputs='true', mlflow_enabled='true')


DEFAULTS = _defaults()
try:
    for key, value in DEFAULTS.items():
        dbutils.widgets.text(key, value)
    get = dbutils.widgets.get
except NameError:
    def get(name):
        return os.environ.get(name.upper(), DEFAULTS[name])

repo = Path(get('repo_root')).resolve()
for filename in ('features.py', 'features_v1.py', 'gate_eval.py'):
    if not (repo / 'ml' / filename).is_file():
        raise FileNotFoundError(f'Check repo_root: missing existing module {repo / "ml" / filename}')
sys.path.insert(0, str(repo / 'ml'))
for name in ('features', 'features_v1', 'gate_eval'):
    sys.modules.pop(name, None)
importlib.invalidate_caches()
from features import FEATURES, check_estimates
from features_v1 import FEATURES_V1, GROUPS, build_table_v1, check_source_estimates
from gate_eval import PARAMS, cutoff, fit, split_ids

## Feature definitions

Only the model-input registry controls training and inference columns. SA values may be present in
the offline table for reference evaluation, but are explicitly excluded from both learned predictors.
Both variants use the same query-ID cohorts, target, LightGBM parameters, and training-only early stopping.

In [ ]:
def variants():
    without_sa = [name for name in FEATURES if name not in GROUPS['skew']]
    result = {'v0_no_skew': without_sa,
              'v0_no_skew_src_contrast': without_sa + ['src_edge_contrast']}
    if any({'skew_pred', 'log_alpha'} & set(features) for features in result.values()):
        raise AssertionError('SA inputs must not enter a no-skew model')
    return result

## Input, split, and artifact helpers

The existing query-ID split is reused. Output directories must be outside the Git checkout and
input roots. Every execution creates a fresh directory; earlier runs are not overwritten.

In [ ]:
def _bool(value):
    if isinstance(value, bool):
        return value
    if str(value).lower() not in ("true", "false"):
        raise ValueError("boolean settings must be true or false")
    return str(value).lower() == "true"


def _jsonable(value):
    if isinstance(value, dict):
        return {str(k): _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(v) for v in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, Path):
        return str(value)
    return value


def _write_json(path, value):
    Path(path).write_text(json.dumps(_jsonable(value), indent=2, allow_nan=False), encoding="utf-8")


def _hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def validate_output_root(output_root, repo_root, data_root, sample_root):
    if not str(output_root).strip():
        raise ValueError("Set output_root to a personal results directory before running")
    output = Path(output_root).expanduser().resolve()
    if output.is_relative_to(Path("/Workspace/Shared")):
        raise ValueError("output_root must be personal, not /Workspace/Shared")
    for protected in (repo_root, data_root, sample_root):
        if output.is_relative_to(Path(protected).resolve()):
            raise ValueError("output_root must be outside the checkout and input directories")
    return output


def make_split_manifest(df, seed):
    train, cal, test = split_ids(df.qId, seed)
    early = set(sorted(train)[::10])
    return pd.DataFrame([
        {"seed": seed, "qId": int(qid), "split": split, "early_stopping": qid in early}
        for split, ids in (("train", train), ("calibration", cal), ("test", test))
        for qid in sorted(ids)
    ])


def _auc(error, score, eps):
    bad = np.asarray(error) > eps
    return float(roc_auc_score(bad, score)) if len(np.unique(bad)) == 2 else np.nan


def _inputs(data_dir, run_dir):
    files = [data_dir / "queries-Ans.txt", data_dir / "queries-meta.csv", run_dir / "rows.csv"]
    for path in files:
        if not path.is_file():
            raise FileNotFoundError(f"Missing input: {path}. Git clone does not create logger outputs.")
    sources = sorted(run_dir.glob("src_*.txt"))
    if not sources:
        raise FileNotFoundError(f"Missing source histogram exports: {run_dir}/src_*.txt")
    rows = pd.read_csv(run_dir / "rows.csv", usecols=["method", "lonRes"])
    resolutions = rows.loc[rows.method == "ARM", "lonRes"].unique()
    if not len(resolutions):
        raise ValueError("rows.csv has no ARM records")
    files += sources + [run_dir / f"arm_{int(res)}.txt" for res in sorted(resolutions)]
    for path in files:
        if not path.is_file():
            raise FileNotFoundError(f"Missing histogram export: {path}")
    return files


def _snapshot(files, data_dir, run_dir, destination, copy_inputs):
    manifest = []
    for source in files:
        before = source.stat()
        target = source
        if copy_inputs:
            kind = "data" if source.parent == data_dir else "run"
            target = destination / "inputs" / kind / source.name
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source, target)
        checksum = _hash(target)
        after = source.stat()
        if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):
            raise RuntimeError(f"Input changed during snapshot: {source}")
        manifest.append(dict(source=str(source), used_path=str(target), size_bytes=before.st_size, sha256=checksum))
    return manifest

## Three calibration policies

**Empirical** reproduces the earlier calibration rule: choose the largest cutoff whose observed
calibration violation fraction is at most delta. It has no post-selection statistical risk guarantee.

**grid_empirical** uses the same TRAIN-fixed candidates as the conservative policy, but only requires
a nonempty accepted calibration set and observed violation fraction <=delta. It has no statistical
certificate. This control separates the effect of using a finite grid from the added risk-bound test.

**ucb_grid** first fixes at most K finite cutoffs from TRAIN score quantiles, including complete ties.
Calibration then measures accepted/violating counts for those fixed cutoffs. A candidate is certified
only if a one-sided Clopper-Pearson upper bound is at most delta. Select the largest certified cutoff.
If none is certified, reject all; do not silently substitute the empirical cutoff.

Before fitting/evaluation, reserve the error budget across the full configured family:

`F = variants × seeds × tree budgets × epsilons × merge rates × 2 scorers × K`

Each candidate uses alpha `(1 - confidence) / F`. Duplicate quantiles and duplicate SA references
do not shrink this predeclared budget. The default authors_range experiment has F=6912 when there
are 18 merge rates. With zero observed violations, roughly 231 accepted calibration cases are needed
to certify 5% risk at that allocation. Rejection of every candidate on a small fixture is expected.

The binomial calculation assumes **independent queries within each rate, a frozen predictor, and an
unchanged query population**. The union allocation does not require independence across rates/seeds.
Repeated queries from different rates are never pooled as independent binomial trials. Confidence
concerns population violation probability conditional on acceptance, not a promise that every finite
test batch has an observed violation fraction <=5%. Repeated manual searches/reruns and distribution
shift are not covered by this run's family. The guarantee population is restricted to exact counts >=10.

In [ ]:
"""Fixed-grid, calibration-only conditional-risk checks for notebook 08.

Each candidate cutoff must be fixed using training information only. Within a
merge rate, calibration observations must represent distinct independent query
samples from the intended unchanged query population. Repeated query rows from
multiple merge rates must not be pooled as independent binomial trials.

The caller supplies a preallocated per-candidate error budget. Bonferroni over
the entire predeclared family then permits choosing any certified candidate,
even if the candidate decisions or merge rates are dependent. The bound targets
population violation probability conditional on acceptance, not a guarantee that
every finite test batch has an observed violation fraction below delta.
"""

import numpy as np
import pandas as pd
from scipy.stats import beta


def _finite_vector(value, name, *, nonempty=False):
    try:
        array = np.asarray(value, dtype=float)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{name} must be a finite numeric vector") from error
    if array.ndim != 1 or (nonempty and len(array) == 0) or not np.isfinite(array).all():
        raise ValueError(f"{name} must be a finite one-dimensional vector"
                         + (" with at least one value" if nonempty else ""))
    return array


def training_threshold_grid(train_score, grid_size):
    """Return at most K finite cutoffs from TRAIN scores, with complete ties.

    No calibration/test scores or errors enter this function. K is the fixed
    maximum family size even when duplicate quantiles leave fewer cutoffs. The
    finite training maximum is a candidate; accept-all infinity is not added.
    Reject-all is implicit and needs no statistical certification.
    """
    if (isinstance(grid_size, (bool, np.bool_))
            or not isinstance(grid_size, (int, np.integer)) or grid_size < 2):
        raise ValueError("grid_size must be an integer >= 2")
    scores = _finite_vector(train_score, "train_score", nonempty=True)
    return np.unique(np.quantile(scores, np.linspace(0.0, 1.0, int(grid_size))))


def calibration_grid_candidates(cal_score, cal_error, candidates, epsilon, delta,
                                alpha_per_candidate):
    """Audit training-fixed cutoffs using one-sided Clopper-Pearson bounds.

    Risk means P(relative error > epsilon | score <= cutoff). Conditional on
    accepted sample size, the binomial calculation needs independent queries
    and a frozen model/cutoff under the population assumptions above. The
    supplied alpha must already account for all configured candidate tests.
    A cutoff with no accepted calibration observations is not certified.
    """
    score = _finite_vector(cal_score, "cal_score")
    error = _finite_vector(cal_error, "cal_error")
    thresholds = _finite_vector(candidates, "candidates", nonempty=True)
    if score.shape != error.shape:
        raise ValueError("cal_score and cal_error must have the same shape")
    if (error < 0).any():
        raise ValueError("cal_error must be nonnegative relative errors")
    try:
        epsilon, delta, alpha = float(epsilon), float(delta), float(alpha_per_candidate)
    except (TypeError, ValueError) as issue:
        raise ValueError("epsilon, delta and alpha must be finite numbers") from issue
    if not np.isfinite(epsilon) or epsilon <= 0:
        raise ValueError("epsilon must be finite and positive")
    if not np.isfinite(delta) or not 0 <= delta < 1:
        raise ValueError("delta must be finite and in [0, 1)")
    if not np.isfinite(alpha) or not 0 < alpha < 1:
        raise ValueError("alpha_per_candidate must be finite and in (0, 1)")
    rows = []
    for threshold in np.unique(thresholds):
        accepted = score <= threshold
        n = int(accepted.sum())
        k = int(((error > epsilon) & accepted).sum())
        observed = k / n if n else np.nan
        # isf(alpha, ...) equals ppf(1-alpha, ...) while avoiding loss of
        # precision when 1-alpha rounds to 1 for very small family budgets.
        upper = float(beta.isf(alpha, k + 1, n - k)) if n and k < n else (1.0 if n else np.nan)
        rows.append(dict(
            cutoff=float(threshold), calibration_accepted_count=n,
            calibration_violation_count=k, calibration_violation=observed,
            calibration_risk_upper=upper,
            calibration_certified=bool(n > 0 and np.isfinite(upper) and upper <= delta),
            alpha_per_candidate=alpha,
        ))
    return pd.DataFrame(rows)


def select_grid_cutoff(candidate_frame):
    """Choose the largest certified cutoff, or reject all with -infinity.

    All candidates were audited with simultaneous error allocation. Selection
    need not assume the conditional error rate increases with the cutoff.
    """
    required = {"cutoff", "calibration_certified"}
    if not isinstance(candidate_frame, pd.DataFrame) or not required <= set(candidate_frame.columns):
        raise ValueError("candidate_frame is missing cutoff/certification columns")
    values = _finite_vector(candidate_frame.cutoff.to_numpy(), "candidate_frame.cutoff")
    certified = candidate_frame.calibration_certified
    if certified.isna().any() or not certified.map(lambda x: isinstance(x, (bool, np.bool_))).all():
        raise ValueError("calibration_certified must contain booleans")
    selected = values[certified.to_numpy(dtype=bool)]
    return float(selected.max()) if len(selected) else -np.inf

## Per-rate evaluation and audit

Every rate must contain distinct query IDs in each cohort. Training alone determines the candidate
grid; calibration alone selects cutoffs; test outcomes only produce the reported quality metrics.
Zero accepted answers have undefined observed risk and do not count as target success.

In [ ]:
def evaluate_rate_scores(train, cal, test, train_scores, cal_scores, test_scores,
                         epsilons, delta, alpha_each, grid_size):
    parts = (train, cal, test)
    score_sets = (train_scores, cal_scores, test_scores)
    rates = sorted(train.mergeRate.unique())
    if any(not len(part) or set(part.mergeRate.unique()) != set(rates) for part in parts):
        raise ValueError('Every cohort must contain the same nonempty merge-rate set')
    if any(part.duplicated(['qId', 'mergeRate']).any() for part in parts):
        raise ValueError('Per-rate binomial evaluation requires distinct query IDs')
    for part, scores in zip(parts, score_sets):
        if set(scores) != set(SCORERS):
            raise ValueError('Expected learned and skew_align score vectors')
        for score in scores.values():
            if np.asarray(score).shape != (len(part),) or not np.isfinite(score).all():
                raise ValueError('Scores must be finite vectors aligned with their cohort')
    details, audits = [], []
    for scorer in SCORERS:
        for epsilon in epsilons:
            for rate in rates:
                tr = train.mergeRate.to_numpy() == rate
                ca = cal.mergeRate.to_numpy() == rate
                te = test.mergeRate.to_numpy() == rate
                if not tr.any() or not ca.any() or not te.any():
                    raise ValueError('A rate has an empty train/calibration/test cohort')
                train_score = np.asarray(train_scores[scorer])[tr]
                cal_score = np.asarray(cal_scores[scorer])[ca]
                test_score = np.asarray(test_scores[scorer])[te]
                cal_error, test_error = cal.relErr.to_numpy()[ca], test.relErr.to_numpy()[te]
                grid = training_threshold_grid(train_score, grid_size)
                candidate = calibration_grid_candidates(cal_score, cal_error, grid,
                                                        epsilon, delta, alpha_each)
                candidate.insert(0, 'candidate_id', np.arange(len(candidate)))
                audits.append(candidate.assign(score=scorer, eps=epsilon, mergeRate=rate,
                                                training_count=int(tr.sum())))
                grid_ok = candidate[(candidate.calibration_accepted_count > 0)
                                    & (candidate.calibration_violation <= delta)]
                grid_empirical = float(grid_ok.cutoff.max()) if len(grid_ok) else -np.inf
                thresholds = {'empirical': float(cutoff(cal_score, cal_error, epsilon, delta)),
                              'grid_empirical': grid_empirical, 'ucb_grid': select_grid_cutoff(candidate)}
                for policy, threshold in thresholds.items():
                    accepted_cal, accepted_test = cal_score <= threshold, test_score <= threshold
                    n_cal = int(accepted_cal.sum())
                    k_cal = int(((cal_error > epsilon) & accepted_cal).sum())
                    n_test = int(accepted_test.sum())
                    k_test = int(((test_error > epsilon) & accepted_test).sum())
                    risk = k_test / n_test if n_test else np.nan
                    certified, upper = None, np.nan
                    if policy == 'ucb_grid':
                        certified = bool(math.isfinite(threshold))
                        if certified:
                            chosen = candidate.loc[candidate.cutoff == threshold].iloc[0]
                            upper = float(chosen.calibration_risk_upper)
                    details.append(dict(
                        score=scorer, policy=policy, eps=float(epsilon), scope='per_rate', mergeRate=float(rate),
                        cutoff=threshold if math.isfinite(threshold) else None,
                        accept_none=not math.isfinite(threshold), training_count=int(tr.sum()),
                        calibration_count=int(ca.sum()), calibration_accepted_count=n_cal,
                        calibration_violation_count=k_cal, calibration_violation=k_cal / n_cal if n_cal else np.nan,
                        calibration_risk_upper=upper, calibration_certified=certified,
                        alpha_per_candidate=alpha_each if policy == 'ucb_grid' else np.nan,
                        test_count=int(te.sum()), accepted_count=n_test, violation_count=k_test,
                        coverage=n_test / int(te.sum()), violation=risk,
                        quality_target_met=(risk <= delta) if n_test else None,
                        auc=_auc(test_error, test_score, epsilon),
                    ))
    gates = pd.DataFrame(details)
    summaries = []
    for (scorer, policy, epsilon), group in gates.groupby(['score', 'policy', 'eps'], sort=False):
        accepted, bad = int(group.accepted_count.sum()), int(group.violation_count.sum())
        summaries.append(dict(
            score=scorer, policy=policy, eps=epsilon, auc_per_rate=group.auc.mean(),
            coverage_per_rate=group.coverage.mean(), violation_per_rate=group.violation.mean(),
            violation_pooled_per_rate_gates=bad / accepted if accepted else np.nan,
            rates_with_acceptances=int((group.accepted_count > 0).sum()), rates_total=len(group),
            accepted_cases_per_rate=accepted, violating_cases_per_rate=bad,
            calibration_certified_rates=int(group.calibration_certified.eq(True).sum()),
            unique_test_queries=int(test.qId.nunique()), test_cases=len(test),
        ))
    return pd.DataFrame(summaries), gates, pd.concat(audits, ignore_index=True)

## Train once, evaluate three policies

The model predicts log relative error. Lower scores are accepted. The separate `skew_align`
reference uses the logger's SA score and receives the same three calibration policies.
Model artifacts save the exact ordered no-SA feature schema and all three learned-model cutoff policies.
Training, calibration, and test score files allow independent reproduction of grids and decisions.
For a fixed model/scorer, AUC is identical across policies: policies change acceptance, not score ranking.

In [ ]:
def _train(df, features, seed, params):
    if {'skew_pred', 'log_alpha'} & set(features):
        raise ValueError('SA inputs are forbidden in no-skew models')
    tr_ids, ca_ids, te_ids = split_ids(df.qId, seed)
    train, cal, test = (df[df.qId.isin(ids)] for ids in (tr_ids, ca_ids, te_ids))
    if train.qId.nunique() < 3 or not len(cal) or not len(test):
        raise ValueError('Insufficient query IDs; do not resplit to repair the study')
    tick = time.perf_counter()
    model = fit(train, features, seed, params=params)
    fit_seconds = time.perf_counter() - tick
    tick = time.perf_counter()
    scores = [dict(learned=model.predict(part[features]), skew_align=part.skew_pred.to_numpy())
              for part in (train, cal, test)]
    return model, train, cal, test, scores, fit_seconds, time.perf_counter() - tick


def _record_model(df, name, features, seed, budget, plan, folder):
    params = dict(PARAMS, n_estimators=budget, n_jobs=int(plan['config']['num_threads']))
    model, train, cal, test, scores, fit_seconds, predict_seconds = _train(df, features, seed, params)
    tick = time.perf_counter()
    summary, gates, candidates = evaluate_rate_scores(
        train, cal, test, *scores, plan['epsilons'], plan['delta'],
        plan['alpha_per_candidate'], plan['grid_size'])
    evaluate_seconds = time.perf_counter() - tick
    folder.mkdir(parents=True, exist_ok=False)
    model.booster_.save_model(str(folder / 'model.txt'))
    _write_json(folder / 'schema.json', dict(
        features=features, variant=name, seed=seed, tree_budget=budget, params=params,
        best_iteration=model.best_iteration_, min_exact=10,
        forbidden_model_features=['skew_pred', 'log_alpha'],
        label='log(max(abs(estimate-truth)/truth, 0.001))',
        score='predicted log-error; lower is better; cutoffs use this score space'))
    _write_json(folder / 'cutoffs.json', gates[gates.score == 'learned'].to_dict('records'))
    for label, part, score in zip(('training_scores', 'calibration_predictions', 'test_predictions'),
                                  (train, cal, test), scores):
        part[['qId', 'mergeRate', 'relErr']].assign(
            score=score['learned'], skew_align=score['skew_align']).to_csv(folder / (label + '.csv'), index=False)
    early = set(sorted(train.qId.unique())[::10])
    cohorts = pd.concat([part[['qId', 'mergeRate']].assign(split=label)
                         for part, label in ((train, 'train'), (cal, 'calibration'), (test, 'test'))], ignore_index=True)
    cohorts['early_stopping'] = (cohorts.split == 'train') & cohorts.qId.isin(early)
    cohorts.to_csv(folder / 'cohorts.csv', index=False)
    for frame in (summary, gates, candidates):
        frame.insert(0, 'variant', name)
        frame.insert(1, 'tree_budget', budget)
        frame.insert(2, 'seed', seed)
        frame.insert(3, 'n_features', len(features))
    summary['fit_seconds'], summary['prediction_seconds'] = fit_seconds, predict_seconds
    summary['evaluation_seconds'], summary['best_iteration'] = evaluate_seconds, model.best_iteration_
    summary.to_csv(folder / 'results.csv', index=False)
    candidates.to_csv(folder / 'threshold_candidates.csv', index=False)
    candidates[['score', 'eps', 'mergeRate', 'candidate_id', 'cutoff', 'training_count']].to_csv(
        folder / 'training_threshold_grid.csv', index=False)
    importance = pd.DataFrame(dict(feature=features, gain=model.booster_.feature_importance('gain'),
                                   variant=name, tree_budget=budget, seed=seed))
    importance.to_csv(folder / 'importance.csv', index=False)
    return summary, gates, candidates, importance


def _log_mlflow(mlflow, folder, result, config, features, params):
    with mlflow.start_run(run_name=f"{config['dataset']}-{folder.parent.parent.name}-{folder.parent.name}-{folder.name}"):
        mlflow.log_params(dict(dataset=config['dataset'], features=','.join(features), **params))
        mlflow.set_tag('personal_result_dir', str(folder))
        for row in result.itertuples():
            for metric in ('auc_per_rate', 'coverage_per_rate', 'violation_per_rate'):
                value = float(getattr(row, metric))
                if math.isfinite(value):
                    mlflow.log_metric(f'{row.score}_{row.policy}_eps{row.eps}_{metric}', value)
        mlflow.log_metric('fit_seconds', float(result.fit_seconds.iloc[0]))
        mlflow.log_artifacts(str(folder), artifact_path='model_evaluation')

## Study plan and summaries

The maximum candidate family is fixed from configured variants, seeds, tree budgets, epsilons,
merge-rate metadata, scorers and K before looking at calibration outcomes. Every configured model
and policy is reported. Hyperparameter exploration should use training validation; do not pick
the setting that looks best on these test outcomes.

In [ ]:
def plan_experiment(config):
    cfg = dict(config)
    if Path(cfg['repo_root']).resolve() != repo:
        raise ValueError('repo_root changed; rerun the Settings/import cell')
    dataset = str(cfg['dataset']).strip()
    if not dataset or dataset in ('.', '..') or '/' in dataset or '\\' in dataset:
        raise ValueError('dataset must be a single directory name')
    cfg['dataset'] = dataset
    validate_output_root(cfg['output_root'], cfg['repo_root'], cfg['data_root'], cfg['sample_root'])
    names = [name.strip() for name in cfg['variants'].split(',')]
    if not names or len(names) != len(set(names)) or set(names) - variants().keys() or BASELINE_VARIANT not in names:
        raise ValueError(f'Select distinct variants including {BASELINE_VARIANT} from {list(variants())}')
    seeds = [int(value.strip()) for value in str(cfg['seeds']).split(',')]
    budgets = [int(value.strip()) for value in str(cfg['tree_budgets']).split(',')]
    epsilons = [float(value.strip()) for value in str(cfg['epsilons']).split(',')]
    delta, confidence = float(cfg['delta']), float(cfg['confidence'])
    grid_size, threads = int(cfg['grid_size']), int(cfg['num_threads'])
    if not seeds or len(set(seeds)) != len(seeds) or min(seeds) < 0:
        raise ValueError('seeds must be distinct nonnegative integers')
    if not budgets or len(set(budgets)) != len(budgets) or min(budgets) < 1 or threads < 1:
        raise ValueError('tree_budgets must be distinct positive integers; num_threads must be positive')
    if (not epsilons or len(set(epsilons)) != len(epsilons)
            or any(not math.isfinite(e) or e <= 0 for e in epsilons)
            or not 0 <= delta < 1 or not 0 < confidence < 1 or grid_size < 2):
        raise ValueError('Invalid error target, confidence, or grid_size')
    _bool(cfg['copy_inputs'])
    if _bool(cfg['mlflow_enabled']) and not str(cfg['experiment']).startswith('/Users/'):
        raise ValueError('Set a personal /Users/<account>/... MLflow experiment')
    data = Path(cfg['data_root']).resolve() / dataset
    samples = Path(cfg['sample_root']).resolve() / dataset
    _inputs(data, samples)
    metadata = pd.read_csv(samples / 'rows.csv', usecols=['method', 'mergeRate'])
    rates = sorted(metadata.loc[metadata.method == 'ARM', 'mergeRate'].unique())
    if not rates or not np.isfinite(rates).all():
        raise ValueError('Logger metadata must define finite merge rates')
    family = len(names) * len(seeds) * len(budgets) * len(epsilons) * len(rates) * len(SCORERS) * grid_size
    return dict(config=cfg, variants=names, seeds=seeds, tree_budgets=budgets, epsilons=epsilons,
                delta=delta, confidence=confidence, grid_size=grid_size, rates=rates,
                family_hypotheses=family, alpha_per_candidate=(1 - confidence) / family,
                fit_count=len(names) * len(seeds) * len(budgets))


def _tables(results, details, destination, delta):
    keys = ['variant', 'tree_budget', 'score', 'policy', 'eps']
    metrics = ['auc_per_rate', 'coverage_per_rate', 'violation_per_rate',
               'violation_pooled_per_rate_gates', 'fit_seconds']
    grouped = results.groupby(keys)
    summary = grouped[metrics].agg(['mean', 'std'])
    summary.columns = [f'{metric}_{stat}' for metric, stat in summary.columns]
    summary = summary.reset_index().merge(grouped.seed.nunique().rename('n_seed').reset_index())
    for metric in metrics:
        summary = summary.merge(grouped[metric].count().rename(metric + '_n').reset_index())
    counts = grouped[['accepted_cases_per_rate', 'violating_cases_per_rate']].sum().reset_index()
    counts['case_weighted_violation'] = counts.violating_cases_per_rate / counts.accepted_cases_per_rate.where(counts.accepted_cases_per_rate > 0)
    summary = summary.merge(counts, on=keys, validate='one_to_one')
    summary.to_csv(destination / 'summary.csv', index=False)
    rate = details.copy()
    rate['above_target'] = (rate.accepted_count > 0) & (rate.violation > delta)
    rate.groupby(keys + ['mergeRate']).agg(
        n_seed=('seed', 'nunique'), coverage_mean=('coverage', 'mean'), coverage_std=('coverage', 'std'),
        violation_mean=('violation', 'mean'), violation_std=('violation', 'std'),
        defined_risk_seeds=('violation', 'count'), above_target_seeds=('above_target', 'sum'),
        accepted_cases=('accepted_count', 'sum'), violating_cases=('violation_count', 'sum'),
    ).reset_index().to_csv(destination / 'per_rate_summary.csv', index=False)
    compared = ['coverage_per_rate', 'violation_per_rate', 'auc_per_rate',
                'violation_pooled_per_rate_gates', 'accepted_cases_per_rate', 'violating_cases_per_rate']
    paired_keys = ['tree_budget', 'seed', 'score', 'policy', 'eps']
    reference = results[results.variant == BASELINE_VARIANT][paired_keys + compared].rename(
        columns={metric: 'baseline_' + metric for metric in compared})
    paired = results[results.variant != BASELINE_VARIANT].merge(reference, on=paired_keys, validate='many_to_one')
    for metric in compared:
        paired['delta_' + metric] = paired[metric] - paired['baseline_' + metric]
    paired.to_csv(destination / 'paired_deltas.csv', index=False)
    policy_keys = ['variant', 'tree_budget', 'seed', 'score', 'eps']
    empirical = results[results.policy == 'empirical'][policy_keys + compared].rename(
        columns={metric: 'empirical_' + metric for metric in compared})
    policy_delta = results[results.policy != 'empirical'].merge(empirical, on=policy_keys, validate='many_to_one')
    policy_delta['baseline_policy'] = 'empirical'
    for metric in compared:
        policy_delta['delta_' + metric] = policy_delta[metric] - policy_delta['empirical_' + metric]
    policy_delta.to_csv(destination / 'calibration_deltas.csv', index=False)
    return summary


def run_experiment(config):
    started = time.perf_counter()
    plan = plan_experiment(config)
    cfg = plan['config']
    mlflow = None
    if _bool(cfg['mlflow_enabled']):
        import mlflow
        mlflow.set_experiment(cfg['experiment'])
    data = Path(cfg['data_root']).resolve() / cfg['dataset']
    samples = Path(cfg['sample_root']).resolve() / cfg['dataset']
    destination = Path(cfg['output_root']).resolve() / f"{cfg['dataset']}-{STUDY}-{uuid4().hex[:12]}"
    destination.mkdir(parents=True, exist_ok=False)
    state = dict(status='running', study=STUDY, config=cfg, timings={},
                 feature_definitions={name: variants()[name] for name in plan['variants']},
                 calibration_family={key: plan[key] for key in ('family_hypotheses', 'alpha_per_candidate',
                     'confidence', 'grid_size', 'variants', 'seeds', 'tree_budgets', 'epsilons', 'rates')},
                 calibration_assumptions=[
                     'Training fixes every candidate cutoff; calibration alone selects a cutoff.',
                     'Distinct independent queries within each rate and an unchanged query population are assumed.',
                     'A population conditional-risk bound does not guarantee observed risk on every finite test batch.',
                     'The family covers all configured candidates, not later manual searches or distribution shift.',
                     'Only queries with exact counts >=10 are evaluated; repeated rates are never pooled as binomial trials.'],
                 notes=['Offline quality study; feature generation still reads all source histograms and precomputed SA references.',
                        'SA features are forbidden model inputs; actual SA runtime bypass and latency are not measured.',
                        'No automatic test-guided model or policy selection; report all configured settings.',
                        'Zero acceptance has undefined risk. No feasible UCB cutoff rejects all without substituting another policy.',
                        'Queries overlap across seeds/rates; seed SDs are descriptive, not confidence intervals.'])
    try:
        tick = time.perf_counter()
        manifest = _snapshot(_inputs(data, samples), data, samples, destination, _bool(cfg['copy_inputs']))
        _write_json(destination / 'input_manifest.json', manifest)
        state['input_sha256'] = {('data/' if Path(row['source']).parent == data else 'run/') + Path(row['source']).name: row['sha256'] for row in manifest}
        state['timings']['input_preparation_seconds'] = time.perf_counter() - tick
        if _bool(cfg['copy_inputs']):
            data, samples = destination / 'inputs/data', destination / 'inputs/run'
        tick = time.perf_counter()
        frame = build_table_v1(str(samples), str(data))
        if not len(frame) or frame.duplicated(['qId', 'mergeRate']).any():
            raise ValueError('Missing eligible rows or duplicate per-rate query IDs')
        if set(frame.mergeRate.unique()) != set(plan['rates']):
            raise ValueError('Eligible data rates differ from the predeclared metadata family')
        if not np.isfinite(frame[FEATURES_V1 + ['label', 'relErr', 'mergeRate']].to_numpy(float)).all():
            raise ValueError('Nonfinite features or labels; check inputs')
        state.update(rows=len(frame), unique_queries=int(frame.qId.nunique()),
                     dropped_small_exact=frame.attrs.get('dropped_small_exact', 0),
                     estimate_gaps=dict(merged=check_estimates(frame), sources=check_source_estimates(frame)))
        state['timings']['feature_build_seconds'] = time.perf_counter() - tick
        state['package_versions'] = {name: importlib.metadata.version(name) for name in
                                     ('lightgbm', 'numpy', 'pandas', 'scipy', 'scikit-learn')}
        state['python'] = platform.python_version()
        state['code_sha256'] = {name: _hash(repo / 'ml' / name) for name in ('features.py', 'features_v1.py', 'gate_eval.py')}
        state['inline_helper_sha256'] = {}
        for function in (variants, _snapshot, training_threshold_grid, calibration_grid_candidates,
                         select_grid_cutoff, evaluate_rate_scores, _train, _record_model, _tables,
                         plan_experiment, run_experiment):
            try:
                state['inline_helper_sha256'][function.__name__] = hashlib.sha256(inspect.getsource(function).encode()).hexdigest()
            except (OSError, TypeError) as error:
                state.setdefault('inline_helper_hash_unavailable', {})[function.__name__] = type(error).__name__
        try:
            state['code_sha256'][NOTEBOOK_FILENAME] = _hash(repo / 'ml/notebooks' / NOTEBOOK_FILENAME)
        except OSError as error:
            state['notebook_hash_unavailable'] = type(error).__name__
        try:
            state['git_commit'] = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True, stderr=subprocess.DEVNULL).strip()
            state['git_dirty'] = bool(subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain'], text=True, stderr=subprocess.DEVNULL))
        except (OSError, subprocess.CalledProcessError):
            state['git_commit'] = None
        pd.concat([make_split_manifest(frame, seed) for seed in plan['seeds']], ignore_index=True).to_csv(destination / 'splits.csv', index=False)
        _write_json(destination / 'run.json', state)
        results, gates, candidates, importances = [], [], [], []
        for name in plan['variants']:
            for budget in plan['tree_budgets']:
                for seed in plan['seeds']:
                    folder = destination / 'models' / name / f'trees_{budget}' / f'seed_{seed}'
                    result, detail, candidate, importance = _record_model(
                        frame, name, variants()[name], seed, budget, plan, folder)
                    results.append(result)
                    gates.append(detail)
                    candidates.append(candidate)
                    importances.append(importance)
                    if mlflow is not None:
                        params = dict(PARAMS, n_estimators=budget, n_jobs=int(cfg['num_threads']))
                        _log_mlflow(mlflow, folder, result, cfg, variants()[name], params)
                    print(f'{name} trees={budget} seed={seed}: trained; saved {folder}')
        combined, details = pd.concat(results, ignore_index=True), pd.concat(gates, ignore_index=True)
        combined.to_csv(destination / 'by_seed.csv', index=False)
        details.to_csv(destination / 'gate_details.csv', index=False)
        pd.concat(candidates, ignore_index=True).to_csv(destination / 'threshold_candidates.csv', index=False)
        pd.concat(importances, ignore_index=True).to_csv(destination / 'importance.csv', index=False)
        _tables(combined, details, destination, plan['delta'])
        once_per_fit = combined[(combined.score == 'learned') & (combined.policy == 'empirical')
                                & (combined.eps == plan['epsilons'][0])]
        state['timings']['current_fit_seconds'] = float(once_per_fit.fit_seconds.sum())
        state['timings']['experiment_wall_seconds'] = time.perf_counter() - started
        state['status'] = 'complete'
        _write_json(destination / 'run.json', state)
        print('COMPLETE:', destination)
        return destination
    except Exception as error:
        state.update(status='failed', error=f'{type(error).__name__}: {error}')
        _write_json(destination / 'run.json', state)
        raise

## Review the training plan

The default is six fits. All three calibration policies share each fitted model. Every extra tree
budget adds six fits and enlarges the conservative family; confidence is allocated before calibration.
The plan prints the actual merge-rate count and family, without inspecting calibration/test quality.

In [ ]:
CONFIG = {key: get(key) for key in DEFAULTS}
PLAN = plan_experiment(CONFIG)
print('Repo:', repo)
print('Read-only input roots:', CONFIG['data_root'], CONFIG['sample_root'])
print('Personal output root:', CONFIG['output_root'])
print('Seeds:', PLAN['seeds'], 'tree budgets:', PLAN['tree_budgets'], 'new model fits:', PLAN['fit_count'])
print('Rates:', len(PLAN['rates']), 'candidate family:', PLAN['family_hypotheses'],
      'alpha per candidate:', PLAN['alpha_per_candidate'])
print(pd.DataFrame([dict(variant=name, features=len(variants()[name])) for name in PLAN['variants']]).to_string(index=False))

## Execute

Run from the actual input files. Missing data or permissions fail clearly. Re-running creates a new
directory. Previous experiments are never overwritten, and no previous personal model is required.

In [ ]:
if {key: get(key) for key in DEFAULTS} != CONFIG:
    raise ValueError('Settings changed; rerun Review the training plan before training')
result_dir = run_experiment(CONFIG)
print('Results for this study:', result_dir)

## Read and share results

- `summary.csv`: all configured model budgets, scorers and policies; seed means, sample SDs, valid-risk
  counts and case counts. SDs describe overlapping query splits and are not confidence intervals.
- `by_seed.csv`, `gate_details.csv`, `per_rate_summary.csv`: exact accepted/violating counts and observed
  test risk. `calibration_certified` is separate from `quality_target_met` on the test sample.
- `threshold_candidates.csv`: every training-fixed candidate, calibration counts, its allocated alpha,
  CP upper bound and certification decision. No test outcome participates in this table's decisions.
- `paired_deltas.csv`: contrast model minus the 17-feature baseline at the same budget/seed/scorer/policy.
- `calibration_deltas.csv`: each grid policy minus the original empirical policy using the same model
  and test queries. Compare grid_empirical with empirical to see grid effects; compare UCB with
  grid_empirical to see the added conservatism of certification.
- `models/<variant>/trees_<budget>/seed_<seed>/`: saved Booster, feature schema, all three learned-model cutoff
  policies, training/calibration/test score files, query cohorts, candidate audit and importance.
- `run.json`, `input_manifest.json`, `splits.csv`: status, settings, predeclared family, assumptions,
  hashes, versions, timings and reproducibility metadata. Require `status=complete` before reporting a run.

For the same grid, a certified threshold cannot exceed the grid_empirical threshold; their accepted
sets are nested. Comparing with the original empirical policy does not have this nesting guarantee
on unseen test scores, because training and calibration cutoff locations differ.

A negative coverage delta means fewer accepted answers; a positive risk delta means higher observed
violation among accepted answers. Undefined risk stays missing. Report rejection/coverage alongside
quality: a conservative rule may accept nothing, especially with a small calibration sample.
The SA reference may appear repeatedly across variants/budgets; it is the same reference per seed,
not additional independent evidence.

Previously inspected tests remain exploratory. Do not change delta, the grid, model budget or confidence
until the same test passes. Choose methodology without test feedback and confirm final claims on a
reserved workload. A successful conservative population bound still allows finite test-sample variation.
This notebook does not measure online feature costs, beta-CDF savings, I/O, or exact-fallback latency.

In [ ]:
metadata = json.loads((result_dir / 'run.json').read_text())
if metadata['status'] != 'complete':
    raise ValueError('Study is incomplete; inspect run.json')
summary = pd.read_csv(result_dir / 'summary.csv')
deltas = pd.read_csv(result_dir / 'paired_deltas.csv')
calibration_deltas = pd.read_csv(result_dir / 'calibration_deltas.csv')
try:
    display(summary)
    display(deltas)
    display(calibration_deltas)
except NameError:
    print(summary.to_string(index=False))
    print(deltas.to_string(index=False))
print('Result files:', result_dir)
print('Share summary.csv, by_seed.csv, gate_details.csv, per_rate_summary.csv, paired_deltas.csv, calibration_deltas.csv, threshold_candidates.csv and run.json for analysis.')